In [1]:
# Modules to install
# pip install scikit-clean pandas numpy sklearn

# Importo librerías

In [1]:
import numpy as np
import pandas as pd

# My libraries
from filters import *
from noisers.classes import *
from noisers.funcs import *
from testFuncs import *

# Stablish random_seed/state for reproducibility
rs=33

/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Preparo datos

Importo a continuación un conjunto variado de datasets basícos ofrecidos por sklearn:

In [2]:
import pandas as pd
from sklearn.datasets import load_iris, load_wine, load_digits, load_breast_cancer

# 1) Cargar datasets sklearn
raw_datasets = [
    ("iris", load_iris()),
    ("wine", load_wine()),
    ("digits", load_digits()),
    ("breast_cancer", load_breast_cancer()),
]

# 2) Convertir a DataFrames con target en la última columna
dfs = []
dfs_names = []

for name, d in raw_datasets:
    X = pd.DataFrame(d.data, columns=d.feature_names)
    y = pd.Series(d.target, name="target")
    df = pd.concat([X, y], axis=1)  # target al final
    dfs.append(df)
    dfs_names.append(name)

# dfs -> lista de DataFrames
# dfs_names -> lista de nombres en el mismo orden

# (Opcional) ver shapes y columnas
for name, df in zip(dfs_names, dfs):
    print(f"{name:14s} shape={df.shape}  target_col={df.columns[-1]!r}")

iris           shape=(150, 5)  target_col='target'
wine           shape=(178, 14)  target_col='target'
digits         shape=(1797, 65)  target_col='target'
breast_cancer  shape=(569, 31)  target_col='target'


# Filter testing

### Display available filters

In [4]:
print_available_filters()

['ClassificationFilter',
 'ENNFilter',
 'EnsembleFilter',
 'IterativePartitioningFilter',
 'NCNEdit']

## NCAR noise

Testeo a continuación el desempeño de diferentes técnicas de filtrado amplicando un intercambio aleatorio de las etiquetas de la columna target, ello sobre los difderentes datasets antes importados y haciendo uso de diferentes niveles de ruido.

In [5]:
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier


ef = EnsembleFiltering(
    estimators=[
        GaussianNB(),
        RandomForestClassifier(random_state=33),
    ],
    cv=5,
    mode="consensus",
    # threshold=0.75,
    action="remove",
    random_state=33,
)
cf = ClassificationFilter(
    estimator = KNeighborsClassifier()
)
ipf = IterativePartitioningFilter()
enn = ENNFilter()
menn = ENNFilter(mode="menn")
ncne = NCNEdit()

filters = {
    "ef" : ef, 
    "cf":cf,
    "ipf":ipf,
    "enn":enn,
    "menn":menn,
    "ncne":ncne
}

# --- ejecutar TU función exactamente ---
noise_levels = np.arange(0.05,0.3,0.05)

results = {
    filter_name : urlf_test_in_dfs(
        dfs=dfs[:2],
        dfs_names=dfs_names,
        noise_kw=[{"noise_level": i } for i in noise_levels],
        rs=33,
        noiser= URLFNoise(),
        filtr=filters[filter_name],
        model=KNeighborsClassifier(),
        sc=StandardScaler(),
        k_cv=5,
        verbose=0
    ) 
    for filter_name in filters.keys()
}

Cycling through dataframes:  25%|██▌       | 1/4 [00:00<00:01,  2.31it/s]/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 7 members, which is less than n_splits=10.
  warnings.warn(
/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 6 members, which is less than n_splits=10.
  warnings.warn(
/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 7 members, which is less than n_splits=10.
  warnings.warn(
/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 8 members, which is less than n_splits=10.
  warnings.warn(
/home/juamp/Documents/Master/.venv/lib/python3.14/site-packages

KeyboardInterrupt: 

## NAR noise

Testeo a continuación el desempeño de diferentes técnicas de filtrado amplicando un intercambio aleatorio de las etiquetas de la columna target, ello sobre los difderentes datasets antes importados y haciendo uso de diferentes niveles de ruido.

In [3]:
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier


ef = EnsembleFiltering(
    estimators=[
        GaussianNB(),
        RandomForestClassifier(random_state=33),
    ],
    cv=5,
    mode="consensus",
    # threshold=0.75,
    action="remove",
    random_state=33,
)
cf = ClassificationFilter(
    estimator = KNeighborsClassifier()
)
ipf = IterativePartitioningFilter()
enn = ENNFilter()
menn = ENNFilter(mode="menn")
ncne = NCNEdit()

filters = {
    "ef" : ef, 
    "cf":cf,
    "ipf":ipf,
    "enn":enn,
    "menn":menn,
    "ncne":ncne
}

# --- ejecutar TU función exactamente ---
noise_levels = np.arange(0.05,0.3,0.05)

results = {
    filter_name : urlf_test_in_dfs(
        dfs=dfs[:2],
        dfs_names=dfs_names,
        noise_kw=[{"random_state": rs}],
        rs=33,
        noiser= NARNoise(),
        filtr=filters[filter_name],
        model=KNeighborsClassifier(),
        sc=StandardScaler(),
        k_cv=5,
        verbose=0
    ) 
    for filter_name in filters.keys()
}

Cycling through dataframes:  50%|█████     | 2/4 [00:01<00:01,  1.80it/s]
